In [ ]:
from pathlib import Path
import sys

import pandas as pd
import torch
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from torch.utils.data import DataLoader
from torchvision import transforms
from tqdm.auto import tqdm

def find_project_root(start_path: Path, required_dir: str = "src") -> Path:
    current = start_path.resolve()
    while True:
        if (current / required_dir).exists():
            return current
        if current.parent == current:
            raise FileNotFoundError(f"Cannot find project root containing '{required_dir}'")
        current = current.parent

PROJECT_ROOT = find_project_root(Path.cwd(), required_dir="src")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.datasets.multimodal_raw_dataset import (
    MultiModalRawDataset,
    multimodal_raw_collate_fn,
)
from src.models.multimodal.pipeline.pipeline_text_guided_infonce_supcon import (
    MultiModalPipelineTextGuidedInfoNCESupCon,
)

print("PROJECT_ROOT:", PROJECT_ROOT)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

In [ ]:
VAL_IMAGE_ROOT = PROJECT_ROOT / "data" / "AIDG" / "dataset_PlantDoc" / "images" / "validation"
VAL_CAPTION_ROOT = PROJECT_ROOT / "data" / "AIDG" / "captions_LLaVA" / "validation"

CKPT_PATH = PROJECT_ROOT / "archive" / "text_guided_infonce_supcon" / "best_model.pt"

NUM_CLASSES = 28
BATCH_SIZE = 8
IMG_SIZE = 224
NUM_WORKERS = 0

SAVE_DIR = PROJECT_ROOT / "archive" / "text_guided_infonce_supcon" / "validation_results_raw"
SAVE_DIR.mkdir(parents=True, exist_ok=True)

REPORT_PATH = SAVE_DIR / "classification_report.csv"
CM_PATH = SAVE_DIR / "confusion_matrix.csv"
PRED_PATH = SAVE_DIR / "predictions.csv"

print("VAL_IMAGE_ROOT:", VAL_IMAGE_ROOT)
print("VAL_CAPTION_ROOT:", VAL_CAPTION_ROOT)
print("CKPT_PATH:", CKPT_PATH)
print("SAVE_DIR:", SAVE_DIR)

In [ ]:
transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])

In [ ]:
dataset = MultiModalRawDataset(
    image_root=VAL_IMAGE_ROOT,
    caption_root=VAL_CAPTION_ROOT,
    transform=transform,
    use_depth_suppressed=False,
    strict_caption_match=True,
)

loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    collate_fn=multimodal_raw_collate_fn,
    pin_memory=(device == "cuda"),
)

print("Validation samples:", len(dataset))
print("Num classes:", len(dataset.class_to_idx))
print("class_to_idx:", dataset.class_to_idx)

In [ ]:
def main():
    print("Using device:", device)
    print("VAL_IMAGE_ROOT:", VAL_IMAGE_ROOT)
    print("VAL_CAPTION_ROOT:", VAL_CAPTION_ROOT)
    print("CKPT_PATH:", CKPT_PATH)

    model = MultiModalPipelineTextGuidedInfoNCESupCon(
        ckpt_path=CKPT_PATH,
        num_classes=NUM_CLASSES,
        clip_model_name="ViT-L-14",
        clip_pretrained="openai",
        text_input_dim=768,
        image_input_dim=1024,
        proj_dim=512,
        proj_hidden_dim=768,
        pvd_hidden_dim=768,
        cls_hidden_dim=1024,
        dropout=0.3,
        normalize_projection=True,
        device=device,
    ).to(device)

    model.eval()

    y_true = []
    y_pred = []
    prediction_rows = []

    with torch.no_grad():
        for batch in tqdm(loader, desc="Validating raw multimodal"):
            images = batch["image"].to(device, non_blocking=True)
            texts = batch["text"]
            labels = batch["label"].to(device, non_blocking=True)

            logits = model(images, texts)
            preds = torch.argmax(logits, dim=1)

            y_true.extend(labels.cpu().tolist())
            y_pred.extend(preds.cpu().tolist())

            for i in range(len(texts)):
                true_idx = labels[i].item()
                pred_idx = preds[i].item()

                prediction_rows.append({
                    "image_name": batch["image_name"][i],
                    "image_path": batch["image_path"][i],
                    "text": texts[i],
                    "true_label_idx": true_idx,
                    "pred_label_idx": pred_idx,
                    "true_class_name": batch["class_name"][i],
                    "pred_class_name": dataset.idx_to_class[pred_idx],
                    "correct": int(true_idx == pred_idx),
                    "source_json": batch["source_json"][i],
                })

    acc = accuracy_score(y_true, y_pred)
    print(f"Validation Accuracy: {acc:.4f}")

    report = classification_report(
        y_true,
        y_pred,
        target_names=[dataset.idx_to_class[i] for i in range(len(dataset.idx_to_class))],
        digits=4,
        output_dict=True,
        zero_division=0,
    )
    cm = confusion_matrix(y_true, y_pred)

    class_names = [dataset.idx_to_class[i] for i in range(len(dataset.idx_to_class))]
    report_df = pd.DataFrame(report).transpose()
    cm_df = pd.DataFrame(cm, index=class_names, columns=class_names)
    pred_df = pd.DataFrame(prediction_rows)

    report_df.to_csv(REPORT_PATH, encoding="utf-8-sig")
    cm_df.to_csv(CM_PATH, encoding="utf-8-sig")
    pred_df.to_csv(PRED_PATH, index=False, encoding="utf-8-sig")

    print("Saved classification report to:", REPORT_PATH)
    print("Saved confusion matrix to:", CM_PATH)
    print("Saved predictions to:", PRED_PATH)

In [ ]:
main()